# Land sample data

Copies bundle-synced CSVs from `fixtures/sample-data` into the UC Volume landing path used by bronze Auto Loader.
Runs on the all-purpose cluster (`existing_cluster_id`).

In [ ]:
dbutils.widgets.text("catalog", "actuarial_lff")
dbutils.widgets.text("landing_path", "/Volumes/actuarial_lff/bronze/landing")
dbutils.widgets.text("source_path", "")

catalog = dbutils.widgets.get("catalog")
landing_path = dbutils.widgets.get("landing_path").rstrip("/")
source_path = dbutils.widgets.get("source_path").rstrip("/")

spark.sql(f"USE CATALOG {catalog}")
print(f"source_path={source_path}")
print(f"landing_path={landing_path}")

In [ ]:
import shutil
from pathlib import Path

file_map = {
    "claims_bordereau.csv": "claims",
    "premium_bordereau.csv": "premiums",
    "risk_zone_lookup.csv": "risk_zones",
    "cyclone_events.csv": "cyclone_events",
}

src_root = Path(source_path)
if not src_root.exists():
    # Workspace files are often mounted under /Workspace when source_path is a Workspace path
    alt = Path("/Workspace") / source_path.lstrip("/")
    if alt.exists():
        src_root = alt

for filename, subdir in file_map.items():
    src = src_root / filename
    if not src.exists():
        raise FileNotFoundError(f"Missing sample file: {src}")

    dest_dir = Path(landing_path) / subdir
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / filename
    shutil.copy2(src, dest)
    print(f"Landed {src} -> {dest}")

print("Landing complete.")